# 02. 프롬프트를 보내고 응답 받기 (Messages API)

모든 대화 요청은 `client.messages.create()` 하나로 보낸다. 내부적으로는 `POST /v1/messages` 요청이다.

## 먼저 알아둘 것

- **이 노트북부터 요금이 발생한다.** 목록 조회와 달리 모델이 실제로 실행되므로 입력/출력 토큰만큼 과금된다.
  - `claude-opus-5-5` 기준: 입력 $4 / 출력 $20 (100만 토큰당)
  - 이 노트북을 한 번 전부 실행해도 보통 몇 센트 수준이다.
- **API는 대화를 기억하지 않는다(stateless).** 이어지는 대화를 하려면 이전 대화 내용을 매번 전부 다시 보내야 한다.
- **필수 파라미터는 3개**: `model`, `max_tokens`, `messages`
  - `max_tokens`는 응답 길이의 상한선일 뿐 그만큼 쓰는 것이 아니다. 너무 작게 잡으면 답변이 중간에 잘린다.
- **응답 본문(`response.content`)은 문자열이 아니라 블록의 리스트다.** 텍스트 외에 `thinking` 등 다른 종류의 블록이 섞여 올 수 있으므로 `block.type == "text"`를 확인한 뒤 `.text`를 읽는다.

In [1]:
import anthropic
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()

# 01에서 조회한 모델 ID 중 하나를 사용
MODEL = "claude-opus-5-5"

## 1. 가장 기본적인 요청

`messages`는 대화 기록 리스트다. 각 항목은 `role`(`"user"` 또는 `"assistant"`)과 `content`로 구성되고, 첫 메시지는 반드시 `user`여야 한다.

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=16000,
    messages=[
        {"role": "user", "content": "파이썬의 리스트와 튜플의 차이를 세 줄로 설명해줘."}
    ],
)

# content는 블록 리스트 → text 블록만 골라서 출력
for block in response.content:
    if block.type == "text":    # 응답 블록은 ['thinking', 'text']처럼 text 앞에 다른 블록이 올 수 있다. 그래서 content[0].text로 바로 읽으면 오류가 날 수 있음
        print(block.text)

1. **변경 가능성**: 리스트는 생성 후 요소를 추가·삭제·수정할 수 있는 가변(mutable) 객체이고, 튜플은 한 번 만들면 바꿀 수 없는 불변(immutable) 객체입니다.
2. **문법과 성능**: 리스트는 대괄호 `[1, 2, 3]`, 튜플은 소괄호 `(1, 2, 3)`로 만들며, 튜플이 메모리를 덜 쓰고 생성도 약간 더 빠릅니다.
3. **용도**: 리스트는 내용이 바뀌는 데이터 모음에 쓰고, 튜플은 좌표처럼 고정된 데이터에 쓰며, 요소가 모두 불변이면 딕셔너리 키로도 쓸 수 있습니다.


## 2. 응답 객체 살펴보기

| 속성 | 의미 |
|---|---|
| `response.id` | 요청마다 붙는 고유 ID |
| `response.model` | 실제로 응답한 모델 |
| `response.stop_reason` | 응답이 끝난 이유 (아래 표 참고) |
| `response.content` | 응답 블록 리스트 |
| `response.usage` | 사용한 토큰 수 → 요금 계산 기준 |

### stop_reason 값

| 값 | 의미 |
|---|---|
| `end_turn` | 정상적으로 답변을 마침 |
| `max_tokens` | `max_tokens` 한도에 걸려 **잘림** → 값을 늘려야 함 |
| `stop_sequence` | 지정한 중단 문자열을 만남 |
| `tool_use` | 도구 호출을 요청함 (도구 사용 실습에서 다룸) |
| `refusal` | 안전 정책으로 답변을 거절함 → `response.stop_details`에 이유 |

응답을 사용하기 전에 `stop_reason`을 먼저 확인하는 습관을 들이는 것이 좋다.

In [7]:
print("id          :", response.id)
print("model       :", response.model)
print("stop_reason :", response.stop_reason)
print("블록 종류    :", [block.type for block in response.content])

# 토큰 사용량과 대략적인 요금 (claude-opus-5-5: 입력 $4 / 출력 $20 per 1M tokens)
usage = response.usage
cost = usage.input_tokens * 4 / 1_000_000 + usage.output_tokens * 20 / 1_000_000

print(f"입력 토큰   : {usage.input_tokens}")
print(f"출력 토큰   : {usage.output_tokens}")
print(f"  └ thinking: {usage.output_tokens_details.thinking_tokens}")  # 출력 토큰 중 thinking에 쓴 양 (03에서 자세히)
print(f"예상 요금   : ${cost:.5f}")

id          : msg_011Cg9HiWKVPVc2xg1G5JE3z
model       : claude-opus-5-5
stop_reason : end_turn
블록 종류    : ['text']
입력 토큰   : 353
출력 토큰   : 36
  └ thinking: 0
예상 요금   : $0.00213


> **블록 종류에 `thinking`이 보이는 이유**
> Claude Opus 5.5는 답변 전에 항상 내부적으로 생각(thinking)을 할 수 있다(끌 수 없음). 생각 내용은 기본적으로 비어 있는 블록으로 오지만, 생각에 쓴 토큰도 **출력 토큰으로 과금**된다. 다만 간단한 질문에서는 thinking 비중이 작고, 출력 토큰의 대부분은 답변 자체다(한국어는 약 1글자 = 1토큰이고, 마크다운·코드 기호도 토큰을 쓴다). thinking에 쓴 토큰 수는 `usage.output_tokens_details.thinking_tokens`로 확인할 수 있다.

## 3. system 프롬프트와 effort

- `system`: 모델의 역할, 말투, 규칙을 정한다. `messages`와 별도 파라미터로 전달한다.
- `output_config={"effort": ...}`: 얼마나 깊게 생각할지 정한다. `low` / `medium` / `high` / `xhigh` / `max`
  - Claude Opus 5.5의 기본값은 `medium`
  - 낮을수록 빠르고 저렴하고, 높을수록 어려운 문제에서 정확해진다. 간단한 질문은 `low`로 충분하다.

In [4]:
response = client.messages.create(
    model=MODEL,
    max_tokens=16000,
    system="너는 초등학생에게 설명하는 친절한 선생님이야. 쉬운 단어만 쓰고 비유를 들어 설명해.",
    output_config={"effort": "low"},    # effort는 생각의 양을 조절할 뿐이고 답변 길이는 프롬프트가 결정    
                                        # 짧은 답을 원하면 명시해야함 ex) 3줄로 답변해줘
    messages=[
        {"role": "user", "content": "인터넷은 어떻게 작동해?"}
    ],
)

for block in response.content:
    if block.type == "text":    
        print(block.text)

print(f"\n[출력 토큰: {response.usage.output_tokens}]")

## 인터넷은 어떻게 작동할까? 🌍

좋은 질문이에요! 인터넷을 **세상에서 제일 큰 우체국**이라고 생각해 볼까요?

### 📮 1. 편지를 보내는 것과 비슷해요

여러분이 유튜브에서 고양이 영상을 보고 싶다고 해 봐요. 그러면 컴퓨터가 유튜브에게 이렇게 **편지**를 보내요.

> "안녕하세요! 고양이 영상 좀 보내 주세요!"

그러면 유튜브가 **답장**으로 고양이 영상을 보내 줘요. 이 일이 눈 깜짝할 사이에 일어나요! ⚡

### 🏠 2. 모든 컴퓨터에는 주소가 있어요

우리 집에 주소가 있어야 택배가 오잖아요? 인터넷에 연결된 컴퓨터와 스마트폰에도 **주소**가 있어요. 그래서 편지가 길을 잃지 않고 정확한 곳으로 찾아갈 수 있어요.

### 🧩 3. 큰 편지는 조각조각 나눠서 보내요

영상처럼 큰 것은 한 번에 보내기 어려워요. 그래서 **퍼즐 조각처럼** 작게 나눠서 보내요. 조각들이 도착하면 컴퓨터가 퍼즐을 맞추듯 다시 하나로 합쳐요. 🧩➡️🖼️

### 🛣️ 4. 길은 어떻게 생겼을까요?

편지 조각들은 **보이지 않는 길**을 따라 달려가요.
- 땅속과 바닷속에 아주 긴 **전선(케이블)** 이 깔려 있어요.
- 와이파이는 **눈에 안 보이는 전파**로 편지를 실어 날라요. 📶

### 🚦 5. 길 안내를 해 주는 친구도 있어요

길이 갈라지는 곳마다 **교통경찰** 같은 기계(공유기, 라우터)가 있어요. "이 편지는 저쪽으로 가세요!" 하고 길을 알려 줘요.

---

### ✨ 정리해 볼까요?

| 인터넷 | 비유 |
|---|---|
| 데이터 | 편지 📩 |
| 주소 | 집 주소 🏠 |
| 데이터 조각 | 퍼즐 조각 🧩 |
| 케이블, 와이파이 | 길 🛣️ |
| 라우터 | 교통경찰 🚦 |

결국 인터넷은 **전 세계 컴퓨터들이 아주 빠르게 편지를 주고받는 거대한 우체국**이에요! 😊

궁금한 게 더 있으면 언제든지 물어보세요! 🙋

[출력 토큰: 1021]


## 4. 이어지는 대화 (멀티턴)

API는 이전 요청을 기억하지 않으므로, 대화 기록 리스트를 직접 관리하며 매번 전체를 보낸다.

- 모델의 답변은 `response.content` **전체를 그대로** `assistant` 메시지로 추가한다. 텍스트만 뽑아서 넣으면 thinking 블록이 빠져 이후 응답 품질에 영향을 줄 수 있다.
- 대화가 길어질수록 매 요청의 입력 토큰(=요금)도 늘어난다.

In [ ]:
# 모델은 대화를 기억하지 못한다
# messages 리스트에 계속 append해서 이전 맥락을 파악할 수 있도록
# 결국 대화가 길어질 수록 토큰 소모가 심해짐

messages = [
    {"role": "user", "content": "내 이름은 건호고, 좋아하는 과일은 귤이야."}
]

# 첫 번째 요청
response = client.messages.create(model=MODEL, max_tokens=16000, messages=messages)
messages.append({"role": "assistant", "content": response.content})

# 두 번째 요청: 이전 대화를 포함해서 다시 보냄
messages.append({"role": "user", "content": "내 이름과 좋아하는 과일이 뭐였지?"})
response = client.messages.create(model=MODEL, max_tokens=16000, messages=messages)
messages.append({"role": "assistant", "content": response.content})

for block in response.content:
    if block.type == "text":
        print(block.text)

print(f"\n[대화 기록 수: {len(messages)}, 이번 요청 입력 토큰: {response.usage.input_tokens}]")

건호 님이고, 좋아하는 과일은 귤이라고 하셨어요! 🍊

[대화 기록 수: 4, 이번 요청 입력 토큰: 353]


## 5. 에러와 거절(refusal) 처리

실패에는 두 종류가 있다.

1. **요청 자체가 실패** → 예외(exception) 발생. `try/except`로 처리
   - `BadRequestError`(400): 파라미터가 잘못됨
   - `AuthenticationError`(401): API Key 문제
   - `NotFoundError`(404): 모델 ID 오타 등
   - `RateLimitError`(429): 요청이 너무 많음 (SDK가 자동 재시도 후에도 실패한 경우)
2. **요청은 성공했지만 모델이 거절** → 예외 없이 `stop_reason == "refusal"`로 응답

### fallbacks (거절 시 다른 모델로 자동 재시도)

Claude Opus 5.5는 일부 주제(사이버 보안, 생물학 등)에서 안전 분류기가 요청을 거절할 수 있다. 정상적인 요청이 잘못 걸리는 경우도 있다.
`fallbacks="default"`를 지정하면 거절되었을 때 **같은 요청 안에서** Anthropic이 권장하는 다른 모델로 자동 재시도한다.

- 베타 기능이라 `client.beta.messages.create()`와 `betas=["server-side-fallback-2026-07-01"]`를 함께 써야 한다.
- 재시도가 일어나면 `response.model`이 원래 요청한 모델과 달라진다.
- 일반적인 질문에서는 거절이 거의 일어나지 않으므로, 아래 예제는 대부분 그대로 `claude-opus-5-5`가 답한다.

In [ ]:
def ask(prompt: str) -> None:
    try:
        # 모델이 안전 정책에 걸려 대답을 거절하면, 같은 요청 안에서 다른 모델이 대신 답하게 함
        # 거절 판단 기준이 모델마다 다르다 -> 요청 자체가 아니라 '내가 이 능력을 이 요청에 써도 되는건가?'
        # 지금은 베타라서 client.beta. ... 처럼 진행해야함. 추후 수정해야 할 수도 있음
        response = client.beta.messages.create(
            model=MODEL,
            max_tokens=16000,
            betas=["server-side-fallback-2026-07-01"],
            fallbacks="default",
            messages=[{"role": "user", "content": prompt}],
        )
    except anthropic.NotFoundError:
        print(f"모델을 찾을 수 없음: {MODEL}")
        return
    except anthropic.BadRequestError as e:
        print(f"잘못된 요청: {e.message}")
        return

    # 응답 내용을 읽기 전에 stop_reason부터 확인
    if response.stop_reason == "refusal":
        category = response.stop_details.category if response.stop_details else None
        print(f"거절됨 (category: {category})")
        return
    if response.stop_reason == "max_tokens":
        print("[경고] max_tokens 한도에 걸려 답변이 잘렸습니다.\n")

    for block in response.content:
        if block.type == "text":
            print(block.text)
    print(f"\n[응답 모델: {response.model}]")


ask("HTTP와 HTTPS의 차이를 한 문단으로 설명해줘.")

HTTP(HyperText Transfer Protocol)는 웹 브라우저와 서버가 데이터를 주고받기 위한 통신 규약이지만, 데이터를 평문 그대로 전송하기 때문에 중간에서 누군가 통신을 가로채면 비밀번호나 개인정보 같은 내용을 그대로 볼 수 있고 변조될 위험도 있습니다. 반면 HTTPS(HTTP Secure)는 HTTP에 SSL/TLS 암호화 계층을 더한 프로토콜로, 전송되는 데이터를 암호화해 도청과 변조를 막습니다. 또한 신뢰할 수 있는 인증기관(CA)이 발급한 인증서로 접속한 서버가 진짜인지 확인해 피싱 같은 위장 공격도 줄여 줍니다. 일반적으로 HTTP는 80번 포트를, HTTPS는 443번 포트를 기본으로 사용하며, HTTPS는 보안성이 높을 뿐 아니라 검색 엔진 최적화(SEO)에도 유리하고 최신 브라우저가 HTTP 사이트에 '주의 요함' 경고를 표시하기 때문에 오늘날 대부분의 웹사이트에서 표준으로 쓰이고 있습니다.

[응답 모델: claude-opus-5-5]
